# EdgentRAG model services on Lightning AI

Runs embedding (8001), speech-to-text (8002), and generation (8003) in one Studio. Exposes ports with the Lightning SDK and verifies public health URLs. Model names, devices, and limits are read from the committed `app/backend/config.yml` profile.

Provide `EDGENTRAG_EMBEDDING_API_TOKEN`, `EDGENTRAG_STT_API_TOKEN`, and `EDGENTRAG_GENERATION_API_TOKEN` in the environment or hidden prompts. Never save token values in notebook cells.

In [ ]:
from pathlib import Path
from getpass import getpass
import json, os, socket, subprocess, sys, time, urllib.error, urllib.request
from urllib.parse import urlsplit
if sys.version_info < (3, 12): raise RuntimeError('Use Python 3.12 or newer.')
try: from lightning_sdk import Studio
except ImportError: raise RuntimeError('Run: %pip install lightning-sdk') from None
PROJECT_DIR = Path(input('Absolute project root (folder containing app/backend): ').strip()).expanduser()
if not PROJECT_DIR.is_absolute(): raise ValueError('Enter an absolute project path.')
for service in ('embedding','stt','generation'):
    if not (PROJECT_DIR / f'app/backend/src/edgentrag/{service}/app.py').is_file(): raise RuntimeError(f'Missing {service} API.')
CONFIG_FILE = PROJECT_DIR / 'app/backend/config.yml'
if not CONFIG_FILE.is_file(): raise RuntimeError(f'Missing committed model profile: {CONFIG_FILE}')
os.environ['EDGENTRAG_CONFIG_FILE'] = str(CONFIG_FILE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{PROJECT_DIR}/app/backend[embedding,generation,stt]'], check=True)
for service in ('embedding','stt','generation'):
    key = f'EDGENTRAG_{service.upper()}_API_TOKEN'; token = os.environ.get(key) or getpass(f'{key}: ')
    if not token or not token.strip(): raise RuntimeError(f'{key} cannot be empty.')
    os.environ[key] = token
from edgentrag.embedding.settings import EmbeddingSettings
from edgentrag.generation.settings import GenerationSettings
from edgentrag.stt.settings import STTSettings
print('Loaded YAML models:', {'embedding': EmbeddingSettings().model_name, 'stt': STTSettings().model_name, 'generation': GenerationSettings().model_name})
print('Project installed; tokens are available (values hidden).')

## Start and warm the APIs

Embedding and generation are warmed with authenticated requests. STT loads lazily on its first `/transcribe` request; its health response confirms configuration.

In [ ]:
model_processes = globals().get('model_processes', {}); SERVICES = {'embedding':8001,'stt':8002,'generation':8003}
def stop_process(p):
    if p.poll() is None:
        p.terminate()
        try: p.wait(timeout=10)
        except subprocess.TimeoutExpired: p.kill(); p.wait(timeout=5)
def health(url, p, log, attempts=90):
    for _ in range(attempts):
        if p.poll() is not None: raise RuntimeError(f'Process exited; inspect {log}.')
        try:
            with urllib.request.urlopen(url+'/health', timeout=5) as r: return json.load(r)
        except (urllib.error.URLError, TimeoutError): time.sleep(1)
    raise RuntimeError(f'Health check timed out; inspect {log}.')
def post_json(name, url, route, payload):
    req=urllib.request.Request(url+route,data=json.dumps(payload).encode(),headers={'Authorization':'Bearer '+os.environ[f'EDGENTRAG_{name.upper()}_API_TOKEN'],'Content-Type':'application/json'},method='POST')
    with urllib.request.urlopen(req, timeout=600) as r: return json.load(r)
for name, port in SERVICES.items():
    with socket.socket() as probe:
        try: probe.bind(('127.0.0.1',port))
        except OSError: raise RuntimeError(f'Port {port} is occupied.') from None
started=[]
try:
    for name, port in SERVICES.items():
        log=Path(f'/tmp/edgentrag-{name}.log'); env=os.environ.copy()
        with log.open('w') as stream: p=subprocess.Popen([sys.executable,'-m','uvicorn',f'edgentrag.{name}.app:app','--host','0.0.0.0','--port',str(port)],cwd=PROJECT_DIR,env=env,stdout=stream,stderr=subprocess.STDOUT)
        model_processes[name]=p; started.append(p); print(name, health(f'http://127.0.0.1:{port}',p,log))
except Exception:
    for p in reversed(started): stop_process(p)
    raise
print('embedding ready:',post_json('embedding','http://127.0.0.1:8001','/embed',{'texts':['A test document chunk.']}).get('model'))
print('generation ready:',post_json('generation','http://127.0.0.1:8003','/generate',{'prompt':'Explain semantic search in one sentence.','max_new_tokens':64}).get('model'))
print('stt configured; submit audio to /transcribe to load it.')

## Expose ports and verify public URLs

Existing exposed ports are reused so rerunning does not create duplicate endpoints.

In [ ]:
studio=Studio(); MODEL_URLS={}; existing={}
for item in studio.list_ports(): existing[str(getattr(item,'port',getattr(item,'local_port',item)))]=item
for name, port in SERVICES.items():
    info=existing.get(str(port)) or studio.add_ports(port)[0]; urls=getattr(info,'urls',None)
    if not urls: raise RuntimeError(f'No exposed URL for port {port}.')
    url=urls[0].rstrip('/'); parsed=urlsplit(url)
    if parsed.scheme!='https' or not parsed.netloc or parsed.username or parsed.password or parsed.query or parsed.fragment: raise ValueError(f'Invalid public URL for port {port}.')
    MODEL_URLS[name]=url; req=urllib.request.Request(url+'/health',headers={'Authorization':'Bearer '+os.environ[f'EDGENTRAG_{name.upper()}_API_TOKEN']})
    with urllib.request.urlopen(req,timeout=30) as r: print(name,'public API verified:',json.load(r).get('status'))
for key, value in {'EDGENTRAG_USE_COLAB_FOR_EMBEDDING':'false','EDGENTRAG_USE_COLAB_FOR_LLM':'false','EDGENTRAG_LIGHTNING_EMBEDDING_SERVICE_URL':MODEL_URLS['embedding'],'EDGENTRAG_LIGHTNING_STT_SERVICE_URL':MODEL_URLS['stt'],'EDGENTRAG_LIGHTNING_GENERATION_SERVICE_URL':MODEL_URLS['generation']}.items(): print(f'{key}={value}')

Copy the printed settings and matching tokens into your local `.env`, then restart local API/workers. `STOP_SERVICES = True` stops only these processes.

In [ ]:
STOP_SERVICES=False
if STOP_SERVICES:
    for p in model_processes.values(): stop_process(p)
    MODEL_URLS.clear(); print('Model APIs stopped.')
else: print('Services remain running.')